Build the event panel and window CARs. Abnormal return is the stock total return minus the benchmark, on the same basis. Deletions are not sign-flipped here. `n_rebalances` is the sample size used for intervals.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from rebalance.calendar import TradingCalendar
from rebalance.config import load_research_config, repo_root, resolve_config_path
from rebalance.data.prices import load_fundamentals, load_prices
from rebalance.data.russell import load_announcements
from rebalance.data.securities import load_security_master
from rebalance.events.revisions import event_state_asof
from rebalance.events.timeline import select_tradable_events

root = repo_root()
config = load_research_config(root / "config" / "research.yaml")
calendar = TradingCalendar(config.calendar_name, config.timezone)
master = load_security_master(resolve_config_path(root, config.paths.security_master))
events = load_announcements(
    resolve_config_path(root, config.paths.announcements),
    master,
    timezone=config.timezone,
)
trades, excluded = select_tradable_events(events, calendar)
print("Synthetic fixtures only. Do not read these prints as a Russell result.")
print(f"announcements={len(events)} tradable={len(trades)} excluded={len(excluded)}")


Synthetic fixtures only. Do not read these prints as a Russell result.
announcements=23 tradable=20 excluded=1


In [2]:
from rebalance.reporting.summary import summarize_windows
from rebalance.research.event_study import build_event_panel, window_cars

prices = load_prices(resolve_config_path(root, config.paths.prices), calendar=calendar)
panel, diagnostics = build_event_panel(trades, prices, calendar, config.benchmark_security_id)
cars = window_cars(panel, calendar, config.windows)
summary = summarize_windows(
    cars,
    n_resamples=config.bootstrap_resamples,
    seed=config.bootstrap_seed,
    confidence_level=config.confidence_level,
)
columns = ["action", "window", "n_names", "n_rebalances", "mean_car_event_equal", "event_t_df", "bootstrap_ci_low", "bootstrap_ci_high"]
print(f"panel rows={len(panel)} diagnostics={len(diagnostics)}")
print("CAR is a sum of simple abnormal returns. It is not the portfolio's compounded return.")
display(summary[columns])


panel rows=860 diagnostics=0
CAR is a sum of simple abnormal returns. It is not the portfolio's compounded return.


,action,window,n_names,n_rebalances,mean_car_event_equal,event_t_df,bootstrap_ci_low,bootstrap_ci_high
0,ADD,ann_to_rebalance,10,2,0.022989,1,0.014230,0.031749
1,ADD,ann_plus_2_to_rebalance,10,2,0.016317,1,0.009313,0.023321
2,ADD,ann_plus_5_to_rebalance,10,2,0.027526,1,0.009972,0.045080
3,ADD,reb_minus_5_to_rebalance,10,2,0.017243,1,0.013707,0.020779
4,ADD,rebalance_to_plus_1,10,2,0.000365,1,-0.003530,0.004260
5,ADD,rebalance_to_plus_3,10,2,0.018164,1,0.016201,0.020128
6,ADD,rebalance_to_plus_5,10,2,0.028982,1,0.019274,0.038690
7,ADD,rebalance_to_plus_10,10,2,0.061546,1,0.058616,0.064476
8,ADD,rebalance_to_plus_20,10,2,0.061146,1,0.057002,0.065290
9,DELETE,ann_to_rebalance,10,2,0.001233,1,-0.021886,0.024352
